# Model Explainability (SHAP)

Task 3: built-in feature importance, SHAP summary, force plots for TP / FP / FN, and actionable business recommendations.

Requires models saved by `modeling.ipynb`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_loader import load_fraud_data, load_ip_country
from src.explainability import (
    find_example_indices,
    get_builtin_importance,
    make_explainer,
    plot_builtin_importance,
    plot_shap_force,
    plot_shap_summary,
    shap_values,
    top_shap_drivers,
)
from src.features import engineer_fraud_features, select_model_features_fraud
from src.modeling import load_model, stratified_split
from src.preprocessing import clean_fraud_data

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
MODELS = ROOT / "models"

## Load best e-commerce model + holdout features

In [ ]:
model = load_model(MODELS / "fraud_best_model.joblib")
pre = load_model(MODELS / "fraud_preprocessor.joblib")
feature_names = load_model(MODELS / "fraud_feature_names.joblib")

fraud_path = PROC / "fraud_features.csv"
if fraud_path.exists():
    fraud = pd.read_csv(fraud_path, parse_dates=["signup_time", "purchase_time"])
else:
    fraud = engineer_fraud_features(
        clean_fraud_data(load_fraud_data(raw_dir=RAW), load_ip_country(raw_dir=RAW))
    )

X, y = select_model_features_fraud(fraud)
X_train, X_test, y_train, y_test = stratified_split(X, y)
X_test_t = pd.DataFrame(pre.transform(X_test), columns=feature_names)
y_pred = model.predict(X_test_t)
print(model.__class__.__name__, X_test_t.shape)

## 1. Built-in feature importance (top 10)

In [ ]:
imp = get_builtin_importance(model, feature_names, top_n=10)
display(imp)
plot_builtin_importance(imp, title="XGBoost — Top 10 Feature Importances")
plt.show()

## 2. SHAP summary (global)

In [ ]:
# Background / explain on a sample for speed
sample_n = min(2000, len(X_test_t))
X_sample = X_test_t.sample(sample_n, random_state=42)
y_sample = y_test.loc[X_sample.index] if hasattr(y_test, "loc") else y_test[X_sample.index]

explainer = make_explainer(model, X_sample)
sv = shap_values(explainer, X_sample)

plot_shap_summary(sv, X_sample, feature_names=feature_names, max_display=15)

drivers = top_shap_drivers(sv, feature_names, top_n=5)
print("Top 5 SHAP drivers (mean |SHAP|):")
display(drivers)

## 3. Force plots — TP, FP, FN

In [ ]:
y_pred_full = model.predict(X_test_t)
tp_idx = find_example_indices(y_test, y_pred_full, "tp")
fp_idx = find_example_indices(y_test, y_pred_full, "fp")
fn_idx = find_example_indices(y_test, y_pred_full, "fn")

print(f"TP={len(tp_idx)}, FP={len(fp_idx)}, FN={len(fn_idx)}")

# Explain full test set once for force plots
sv_test = shap_values(explainer, X_test_t)

examples = {
    "True Positive (caught fraud)": int(tp_idx[0]) if len(tp_idx) else None,
    "False Positive (legit flagged)": int(fp_idx[0]) if len(fp_idx) else None,
    "False Negative (missed fraud)": int(fn_idx[0]) if len(fn_idx) else None,
}

for title, i in examples.items():
    if i is None:
        print("No example for", title)
        continue
    print("\n===", title, "index", i, "===")
    plot_shap_force(explainer, sv_test, X_test_t, index=i, feature_names=feature_names)
    plt.show()

## 4. Interpretation

Compare built-in importance vs SHAP mean |value|:

- Built-in importance reflects split gain / frequency in trees.
- SHAP attributes each prediction change to features — better for **individual** decisions and regulated reporting.
- Expect strong drivers among: `time_since_signup`, device/user velocity, purchase timing (`hour_of_day`), and high-risk `country` / `source` encodings.
- Surprises to watch: very high `purchase_value` may not dominate if fraudsters probe with small amounts; shared `device_id` can outweigh amount.

## 5. Business recommendations (≥3)

Connect each recommendation to SHAP / importance findings (refine with your actual top drivers after running cells above):

1. **Rapid post-signup purchases** — If `time_since_signup` ranks among top SHAP drivers, require step-up authentication (OTP / 3-D Secure) for purchases within the first few hours of account creation.
2. **Device / user velocity alerts** — When `device_tx_count` or `user_tx_velocity` push SHAP toward fraud, temporarily throttle the device and queue manual review instead of hard-blocking every alert (reduces false-positive friction).
3. **Geo / channel risk tiers** — Countries and traffic sources with elevated fraud contribution in SHAP summary plots should feed a rules engine (lower limits, extra KYC) rather than relying on the model alone.
4. **Dual-threshold operations** — Use a high-precision threshold for automatic declines (minimize FP customer pain) and a lower threshold for soft challenges / monitoring (catch more FN), informed by force-plot patterns on FP vs FN cases.

These actions translate XAI outputs into measurable reductions in chargebacks while protecting customer trust.